| 컬럼명 | 설명 |
|---|---|
| index | 행 인덱스 |
| course_id | 강의 ID |
| userid_DI | 학습자 식별자 |
| registered | 등록 여부 |
| viewed | 강의를 열어본 여부 |
| explored | 강의를 적극적으로 탐색했는지 여부 |
| certified | 인증서 획득 여부 |
| final_cc_cname_DI | 국가 |
| LoE_DI | 학력 수준 (Level of Education) |
| YoB | 출생연도 |
| gender | 성별 |
| grade | 성적 |
| start_time_DI | 강의 시작일 |
| last_event_DI | 마지막 활동일 |
| nevents | 전체 이벤트 수 |
| ndays_act | 실제 활동 일수 |
| nplay_video | 영상 재생 수 |
| nchapters | 탐색한 챕터 수 |
| nforum_posts | 포럼 게시글 수 |
| roles | 사용자 역할 |
| incomplete_flag | 불완전 데이터 관련 플래그 |

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)


In [ ]:
# 데이터 로드
df = pd.read_csv('../../data/Courses.csv')

In [ ]:
print(df.shape)
df.info()

In [ ]:
# 불필요한 컬럼 제거
df.drop(columns='roles', inplace=True)

# havardX 에서는 거의 90퍼센트가 결측치다.
# MIT 의 경우 50.4퍼센트에 육박. 그 중에서도 viewed = 1 인데도 결측이 36% 존재. => 동영상을 봤는데 기록이 없다. 0으로 대체하면 실제로 영상을 안 본 학생이라고 판단하는 셈. 그런데 그대로 두면 전체 결측치가 73% 정도로 매우 많기 때문에 분석에 활용하기 어렵다고 판단.
# => nplay_video 컬럼은 분석에는 제외하고 필요할 때 참고한다.
df.drop(columns='nplay_video', inplace=True)
print(df.shape)
df.info()

In [ ]:
# 컬럼 전체 소문자 변경
df.columns = df.columns.str.lower()

# 컬럼명 변경
df.rename(columns={'userid_di': 'userid', 'final_cc_cname_di': 'country', 'loe_di': 'loe', 'start_time_di': 'start_time', 'last_event_di': 'last_event'}, inplace=True)

# 결과확인
df.info()

In [ ]:
print(df['incomplete_flag'].isna().sum())

In [ ]:
# str => date 타입 변경
df['start_time'] = pd.to_datetime(df['start_time'])
df['last_event'] = pd.to_datetime(df['last_event'])

# str => float 타입 변경
df['grade'] = pd.to_numeric(df['grade'], errors='coerce')

# float => int 타입 변환
df[['nevents', 'ndays_act', 'nchapters', 'incomplete_flag', 'yob']] = df[['nevents', 'ndays_act', 'nchapters', 'incomplete_flag', 'yob']].astype('Int64')

# data 확인
df.info()

In [ ]:
# 데이터 소문자 변경
df['country'] = df['country'].str.lower()
df['userid'] = df['userid'].str.lower()
df['loe'] = df['loe'].str.lower()
df['course_id'] = df['course_id'].str.lower()
df['gender'] = df['gender'].str.lower()
df.head(5)

In [ ]:
print("\n" + "="*60)
print("기초 통계")
print("="*60)
display(df.describe(include='all').T)

In [ ]:
print("\n" + "="*60)
print("결측치 확인")
print("="*60)

missing_df = pd.DataFrame({
    '결측수': df.isnull().sum(),
    '결측비율(%)': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_df = missing_df[missing_df['결측수'] > 0].sort_values('결측수', ascending=False)

if len(missing_df) > 0:
    print("\n[결측치 현황]")
    display(missing_df)
else:
    print("\n결측치 없음")


In [ ]:
# 결측치 대체
df['grade'] = df['grade'].fillna(0)
df['loe'] = df['loe'].fillna('unknown')
df['gender'] = df['gender'].fillna('unknown')

# loe 그룹별 yob 결측치 수
print("loe 그룹별 yob 결측치 수:")
loe_group_na_count = df[df['yob'].isna()].groupby('loe').size()
if loe_group_na_count.empty:
    print("0개")
else:
    display(loe_group_na_count)

# loe 그룹별 yob 중앙값
print("")
yob_median_by_loe = df.groupby('loe')['yob'].median().reset_index()
yob_median_by_loe.rename(columns={'yob': 'yob_median'}, inplace=True)
print("각 그룹 별 중앙값 확인:")
display(yob_median_by_loe)

# yob 결측에 그룹별로 채워넣기
df = df.merge(yob_median_by_loe, on='loe', how='left')
df.loc[df['yob'].isna(), 'yob'] = df.loc[df['yob'].isna(), 'yob_median']
df['yob'] = df['yob'].astype('Int64')
df.drop(columns='yob_median', inplace=True)

print(f"대체 후 결측 수: {df['yob'].isna().sum()}개")

In [ ]:
# 이상치 대체
# o => unknown
# 개수 확인
print("="*60)
print("gender 값이 o => unknown 대체")
print("="*60)

target_count = df.loc[df['gender'] == 'o', 'gender'].count()
print(f"대체 전 대상 개수: {target_count} 개")

df.loc[df['gender'] == 'o', 'gender'] = 'unknown'
replaced_count = df.loc[df['gender'] == 'unknown', 'gender'].count()

print(f"대체 완료!")
print(f"'unknown' 개수: {replaced_count} 개")

In [ ]:
# grade = 0 & certified = 1 인 값 확인 및 제거
# mhxpc130066792
print("="*60)
print("grade = 0 & certified = 1 인 값 확인 및 제거")
print("="*60)

target_user_count = df.loc[(df['grade'] == 0) & (df['certified'] == 1), 'userid'].count()
print(f"삭제 전: {target_user_count} 개")

target_users = df.loc[(df['grade'] == 0) & (df['certified'] == 1), 'userid']
df = df[~df['userid'].isin(target_users)]

# 제거 여부 확인
exists_count = df.loc[(df['grade'] == 0) & (df['certified'] == 1), 'userid'].count()
print(f"삭제 후: {exists_count} 개")

# 데이터 수 확인
print(df.shape)

In [ ]:
print("="*60)
print("viewed = 0 & explored = 1 인 값 확인 및 제거")
print("="*60)

target_user_count = df.loc[(df['viewed'] == 0) & (df['explored'] == 1), 'userid'].count()
print(f"삭제 전: {target_user_count} 개")

target_users = df.loc[(df['viewed'] == 0) & (df['explored'] == 1), 'userid']
df = df[~df['userid'].isin(target_users)]

# 제거 여부 확인
exists_count = df.loc[(df['viewed'] == 0) & (df['explored'] == 1), 'userid'].count()
print(f"삭제 후: {exists_count} 개")

# 데이터 수 확인
print(df.shape)

In [ ]:
print("="*60)
print("last_event 결측값 대체")
print("="*60)

na_cnt = df['last_event'].isna().sum()
print(f"결측 수: {na_cnt}개")

# duration 파생컬럼 생성
df['duration'] = (df['last_event'] - df['start_time']).dt.days

# start_time 이 last_event 보다 큰 이상치 조회 => last_event - start_time 중앙값 계산에서 제외
excepted_users = df.loc[(df['start_time'] > df['last_event']), 'userid']

# 강의 + 활동 별 (last_event - start_event) 중앙값 대체 (이상치 제외)
group_cols = ['course_id', 'registered', 'viewed', 'explored', 'certified']
df_filtered = df[~df['userid'].isin(excepted_users)]
group_duration = df_filtered.groupby(group_cols)['duration'].median().reset_index()
group_duration.rename(columns={'duration': 'duration_median'}, inplace=True)
display(group_duration)

# last_event 결측치를 그룹 중앙값으로 대체
df = df.merge(group_duration, on=group_cols, how='left')
mask = df['last_event'].isna()
df.loc[mask, 'last_event'] = (
    pd.to_datetime(df.loc[mask, 'start_time']) +
    pd.to_timedelta(df.loc[mask, 'duration_median'], unit='D')
).dt.date
df.drop(columns='duration_median', inplace=True)

# group_duration 중복 확인
print(group_duration.duplicated(subset=group_cols).sum())
print(f"\n대체 후 결측 수: {df['last_event'].isna().sum()}개")
print(df.shape)

In [ ]:
# 13세 미만 사용자 제거
print("="*40)
print("13세 미만 사용자 제거")
print("="*40)
under_12 = df.loc[(pd.to_datetime(df['start_time']).dt.year - df['yob'] < 13), 'userid'].count()
print(f"13세 미만 사용자 수: {under_12}")
df['age'] = pd.to_datetime(df['start_time']).dt.year - df['yob']
df = df[df['age'] >= 13]
print(f"제거완료!")
print(df.shape)

In [ ]:
# 파생컬럼 생성
# course_id 소문자 변경 후 기관/강의코드/강의년도/강의학기 생성
df[['institution', 'course_code', 'year_term']] = df['course_id'].str.split('/', expand=True)
df[['course_year', 'course_term']] = df['year_term'].str.split('_', expand=True)
df['course_term'] = df['course_term'].fillna('unknown')
df.drop(columns='year_term', inplace=True)

# course_id 소문자 기준으로 카테고리 매핑
category_map = {
    'harvardx/cb22x/2013_spring': '문학',
    'harvardx/cs50x/2012': '컴퓨터프로그래밍',
    'harvardx/er22x/2013_spring': '인문학',
    'harvardx/ph207x/2012_fall': '의료서비스',
    'harvardx/ph278x/2013_spring': '환경과학',
    'mitx/14.73x/2013_spring': '경제학',
    'mitx/2.01x/2013_spring': '공학',
    'mitx/3.091x/2012_fall': '화학',
    'mitx/3.091x/2013_spring': '화학',
    'mitx/6.002x/2012_fall': '공학',
    'mitx/6.002x/2013_spring': '공학',
    'mitx/6.00x/2012_fall': '컴퓨터프로그래밍',
    'mitx/6.00x/2013_spring': '컴퓨터프로그래밍',
    'mitx/7.00x/2013_spring': '생물학',
    'mitx/8.02x/2013_spring': '물리학',
    'mitx/8.mrev/2013_summer': '물리학',
}

df['course_category'] = df['course_id'].map(category_map)
df.info()

In [ ]:
# 전처리 완료 데이터 생성
df.to_csv('../../data/preprocessed.csv', index=False)

In [ ]:
processed_data = pd.read_csv('../../data/preprocessed.csv')
print(processed_data.shape)
processed_data.info()